# stack-vs-cat — ex2: interleave two tensors with stack and reshape

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `stack-vs-cat`. Running the final beacon cell reports progress against the `PyTorch: stack vs cat` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: stack vs cat` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`stack-vs-cat`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "stack-vs-cat"
DD_SUBTOPIC = "PyTorch: stack vs cat"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `t.stack` vs `t.cat` — quick refresher

- `t.stack(seq, dim=k)`: every tensor in `seq` has the same shape `S`; result has shape `S` with a new axis of size `len(seq)` INSERTED at position `k`. Rank goes UP by 1.
- `t.cat(seq, dim=k)`: every tensor has the same shape EXCEPT possibly axis `k`; result extends axis `k`. Rank STAYS the same.

**Round-trip identity.** `t.stack(seq, dim=k)` is equivalent to `t.cat([s.unsqueeze(k) for s in seq], dim=k)`. Inverting it: `t.stack(...).unbind(dim=k)` returns the original tuple.

**Interleaving.** Given two equal-length 1-D tensors `a, b`, the interleaved tensor `[a0, b0, a1, b1, ...]` is `t.stack([a, b], dim=1).reshape(-1)`. The stack inserts a 'pair' axis, the reshape walks it in the right order.

### Exercise 2 — interleave two tensors with stack and reshape

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Create
> LO: Create an interleaving operation by composing `t.stack` (insert a pair axis) with `.reshape` (collapse it back) — building a rank-up-then-flatten pipeline rather than picking a single op.
> Keywords: stack, reshape, interleave, rank-change
> ```

**KCs targeted:** `stack-inserts-axis`, `cat-along-existing-axis`

Implement `ex2_interleave(a, b)`. Given two 1-D tensors `a`, `b` of equal length `n`, return the 1-D tensor of length `2n` whose values are `[a[0], b[0], a[1], b[1], ..., a[n-1], b[n-1]]`.

**Required approach** (do NOT use a Python loop or `t.cat` alone):
1. `t.stack([a, b], dim=1)` → shape `(n, 2)`. The new axis stores the (a, b) pair at each position `i`.
2. `.reshape(-1)` (or `.flatten()`) → shape `(2n,)`. Row-major walk visits `(0,0), (0,1), (1,0), (1,1), ...` — which IS the interleaved order.

Why this works: the row-major flatten visits the inner axis FASTEST, so the pair `(a[i], b[i])` is emitted together before moving to `i+1`.

Inputs: two equal-length 1-D tensors of the same dtype.
Output: 1-D tensor of length `2 * a.shape[0]`, same dtype.

In [ ]:
def ex2_interleave(a: Tensor, b: Tensor) -> Tensor:
    """Interleave [a0,b0,a1,b1,...] via stack + reshape."""
    raise NotImplementedError()


def _test_ex2():
    # Tiny correctness check.
    a = t.tensor([1, 2, 3])
    b = t.tensor([10, 20, 30])
    out = ex2_interleave(a, b)
    assert tuple(out.shape) == (6,), f'expected (6,), got {tuple(out.shape)}'
    assert t.equal(out, t.tensor([1, 10, 2, 20, 3, 30])), (
        f'order wrong: got {out}; expected [1,10,2,20,3,30].'
    )

    # dtype propagation.
    a = t.tensor([0.5, 1.5])
    b = t.tensor([0.0, 1.0])
    out = ex2_interleave(a, b)
    assert out.dtype == t.float32
    assert t.equal(out, t.tensor([0.5, 0.0, 1.5, 1.0]))

    # Length scaling.
    n = 100
    a = t.arange(n)
    b = t.arange(n) + 1000
    out = ex2_interleave(a, b)
    assert tuple(out.shape) == (2 * n,)
    # Even positions == a, odd positions == b.
    assert t.equal(out[0::2], a), 'even-position slice must equal a'
    assert t.equal(out[1::2], b), 'odd-position slice must equal b'

    # Edge case — length-0 inputs.
    empty_a = t.tensor([], dtype=t.float32)
    empty_b = t.tensor([], dtype=t.float32)
    out = ex2_interleave(empty_a, empty_b)
    assert tuple(out.shape) == (0,), f'empty-in must yield empty-out, got shape {tuple(out.shape)}'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_interleave(a: Tensor, b: Tensor) -> Tensor:
    return t.stack([a, b], dim=1).reshape(-1)
```

**Why `dim=1` (not `dim=0`).** Stack at `dim=0` would give shape `(2, n)`, and flattening walks ALL of `a` first then ALL of `b` — producing `[a0,a1,...,an-1,b0,b1,...]`, the WRONG order. Stack at `dim=1` puts the pair axis on the INSIDE, so it flattens fastest.

**One-line composition.** This is the canonical stack+reshape idiom — it shows up in:
- Audio stereo channel interleaving.
- Bayer-pattern image deinterleaving (with `unfold`).
- Skip-connection alignment when concatenating into a single buffer.

**Why not `t.cat`.** `cat` only extends an existing axis. To interleave you must FIRST introduce a 'pair' axis (the job of `stack`) and then collapse it. There is no single-call cat that produces interleaved output.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()